# Apache Spark 4.2.1 on Docker — Quickstart

This notebook runs **PySpark 4.2.1** in the `spark-jupyter` container and sends work to the standalone cluster (`spark-master` + 2 workers).

| Service | URL |
|---|---|
| JupyterLab | http://localhost:8888/?token=spark |
| Spark Master UI | http://localhost:8080 |
| Spark History Server | http://localhost:18080 |
| Driver UI (while a job runs) | http://localhost:4040 |

Data and apps are shared with the containers through `./data` and `./apps`.

## 1. Create a Spark session on the cluster

`spark.driver.host=jupyter` tells the workers how to reach this notebook's driver process. Without it the workers cannot connect back to the driver.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Docker Quickstart")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Default parallelism: {spark.sparkContext.defaultParallelism}")

Spark 4.2.1
Master: spark://spark-master:7077
Default parallelism: 4


## 2. Load the sample CSV

`sales.csv` is mounted read-only from the host at `/opt/spark-data/sales.csv`.

In [5]:
sales = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("/opt/spark-data/sales.csv")
    .withColumn("revenue", F.round(F.col("units") * F.col("unit_price"), 2))
)

sales.printSchema()
print(f"Rows: {sales.count()}")
display(sales.limit(10))

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- units: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- revenue: double (nullable = true)

Rows: 48


DataFrame[order_id: int, order_date: date, region: string, product: string, units: int, unit_price: double, revenue: double]

## 3. Aggregate with the DataFrame API

In [7]:
by_region = (
    sales.groupBy("region")
    .agg(
        F.sum("units").alias("units_sold"),
        F.round(F.sum("revenue"), 2).alias("revenue"),
        F.countDistinct("order_id").alias("orders"),
    )
    .orderBy(F.desc("revenue"))
)

display(by_region)

DataFrame[region: string, units_sold: bigint, revenue: double, orders: bigint]

## 4. The same question in Spark SQL

In [8]:
sales.createOrReplaceTempView("sales")

display(
    spark.sql(
        """
        SELECT product,
               SUM(units) AS units_sold,
               ROUND(SUM(units * unit_price), 2) AS revenue
        FROM sales
        GROUP BY product
        ORDER BY revenue DESC
        """
    )
)

DataFrame[product: string, units_sold: bigint, revenue: double]

## 5. Write Parquet to the shared data folder

Output lands in `./data/output/sales_parquet` on the host.

In [9]:
out = "/opt/spark-data/output/sales_parquet"
by_region.coalesce(1).write.mode("overwrite").parquet(out)

roundtrip = spark.read.parquet(out)
print(f"Rows written: {roundtrip.count()}")
display(roundtrip)

Rows written: 4


DataFrame[region: string, units_sold: bigint, revenue: double, orders: bigint]

## 6. Stop the session

Run this when you are finished, then restart the kernel before creating another session.

In [10]:
spark.stop()